In [9]:
import pandas as pd

BHAV_FILE = "BhavCopy_NSE_CM_0_0_0_20260925_F_0000.csv"
CONS_FILE = "ind_niftytotalmarket_list.csv"
OUT_FILE = "nifty750_ohlcv_25sep2026.csv"

bhav = pd.read_csv(BHAV_FILE)
cons = pd.read_csv(CONS_FILE)

# Drop the placeholder rows in the constituent file
cons = cons[~cons["Symbol"].str.startswith("DUMMY")]

# Match on symbol AND series (some stocks trade in both EQ and BL/BE)
df = bhav.merge(
    cons[["Symbol", "Series", "Company Name", "Industry"]],
    left_on=["TckrSymb", "SctySrs"],
    right_on=["Symbol", "Series"],
)

df["Change"] = df["ClsPric"] - df["PrvsClsgPric"]
df["PctChange"] = df["Change"] / df["PrvsClsgPric"] * 100

print(f"Total companies : {len(df)}")
print(f"Went up         : {(df['Change'] > 0).sum()}")
print(f"Went down       : {(df['Change'] < 0).sum()}")
print(f"Unchanged       : {(df['Change'] == 0).sum()}")

advances = (df["Change"] > 0).sum()
declines = (df["Change"] < 0).sum()
print(f"A/D ratio       : {advances / declines:.4f}")
print(f"Net advances    : {advances - declines}")

# Save the 750-stock OHLCV file
out = df[["Symbol", "Company Name", "Industry", "OpnPric", "HghPric",
          "LwPric", "ClsPric", "TtlTradgVol", "PrvsClsgPric",
          "Change", "PctChange"]]
out.columns = ["Symbol", "Company", "Industry", "Open", "High",
               "Low", "Close", "Volume", "PrevClose", "Change", "PctChange"]
out.to_csv(OUT_FILE, index=False)
print(f"Saved: {OUT_FILE}")


Total companies : 750
Went up         : 368
Went down       : 381
Unchanged       : 1
A/D ratio       : 0.9659
Net advances    : -13
Saved: nifty750_ohlcv_25sep2026.csv
